In [1]:
# Significant similarity to audio descriptions. 

import numpy as np
import pandas as pd

scores = np.load('scores/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
print(scores.files)
score_names = ['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
score_files = {}
for name in score_names:
    score_files[name] = scores[name].item()

print(score_files['window_zscores'])

['window_scores', 'window_zscores', 'story_scores', 'story_zscores']
{('wheretheressmoke', 'WER'): array([ 6.25055987e-01,  1.54919334e+00,  1.45774958e+00,  1.33936787e+00,
        1.69937152e+00,  1.61598135e+00,  2.00698861e+00,  1.73152978e+00,
        3.03979376e-01,  1.22500941e-01,  1.06644164e+00,  1.39585843e+00,
        1.81772642e+00,  1.52208010e+00,  9.74185538e-01,  1.04293630e+00,
        3.00865336e-01,  1.58554873e+00,  1.28073887e+00,  1.87501316e+00,
        1.86308593e+00,  7.68941313e-01,  8.31715849e-01,  1.14841431e+00,
        1.20650193e+00,  1.09089769e+00,  2.64386351e-01,  1.59810329e+00,
        1.19344456e+00,  1.62649485e+00,  1.43543122e+00,  1.58586736e+00,
        9.67986391e-01,  1.25797850e+00,  1.51289998e+00,  1.35976906e+00,
        1.68766392e+00,  1.35987570e+00,  9.37911840e-01,  1.17932485e+00,
        1.76532044e-01,  3.83123063e-01,  3.65392048e-02, -1.23869628e-15,
        1.62606351e-01,  1.31032835e+00,  1.10288853e+00,  6.97588692e-01,
 

In [2]:
window_zscores = {'subject': [], 'WER':[],'BLEU':[], 'METEOR':[], 'BERT':[]}
for subject in [1,2,3]:
    for task in ['wheretheressmoke']:
        scores = np.load(f'scores/S{subject}/perceived_speech/{task}.npz', allow_pickle=True)['window_zscores'].item()
        # print(scores['window_zscores'].item())
        window_zscores['subject'].append(subject)
        window_zscores['WER'].append(scores[(task, 'WER')])
        window_zscores['BLEU'].append(scores[(task, 'BLEU')])
        window_zscores['METEOR'].append(scores[(task, 'METEOR')])
        window_zscores['BERT'].append(scores[(task, 'BERT')])

window_zscores

{'subject': [1, 2, 3],
 'WER': [array([ 6.25055987e-01,  1.54919334e+00,  1.45774958e+00,  1.33936787e+00,
          1.69937152e+00,  1.61598135e+00,  2.00698861e+00,  1.73152978e+00,
          3.03979376e-01,  1.22500941e-01,  1.06644164e+00,  1.39585843e+00,
          1.81772642e+00,  1.52208010e+00,  9.74185538e-01,  1.04293630e+00,
          3.00865336e-01,  1.58554873e+00,  1.28073887e+00,  1.87501316e+00,
          1.86308593e+00,  7.68941313e-01,  8.31715849e-01,  1.14841431e+00,
          1.20650193e+00,  1.09089769e+00,  2.64386351e-01,  1.59810329e+00,
          1.19344456e+00,  1.62649485e+00,  1.43543122e+00,  1.58586736e+00,
          9.67986391e-01,  1.25797850e+00,  1.51289998e+00,  1.35976906e+00,
          1.68766392e+00,  1.35987570e+00,  9.37911840e-01,  1.17932485e+00,
          1.76532044e-01,  3.83123063e-01,  3.65392048e-02, -1.23869628e-15,
          1.62606351e-01,  1.31032835e+00,  1.10288853e+00,  6.97588692e-01,
          1.13346614e+00, -4.98630023e-01,  1.

In [3]:
results_df = pd.DataFrame(window_zscores)

print(len(results_df.loc[0, 'WER']))
print(len(results_df.loc[1, 'WER']))
print(len(results_df.loc[2, 'WER']))
print('=')
print(563*3)
m=1689

from scipy.stats import norm
#convert to p val
results_df['WER'] = results_df['WER'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1-norm.cdf(z) for z in l])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1-norm.cdf(z) for z in l])
#sort
for i in range(3):
    for met in ['WER','BLEU','METEOR','BERT']:
        results_df.loc[i,met].sort()
# to q and check threshold
def p_to_q(p, i):
    return p*m/i
results_df['WER'] = results_df['WER'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BLEU'] = results_df['BLEU'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['METEOR'] = results_df['METEOR'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])
results_df['BERT'] = results_df['BERT'].apply(lambda l : [1 if p_to_q(l[i-1], i) < 0.05 else 0 for i in range(1, len(l)+1)])

# S1=np.array(results_df.loc[0, 'BERT'] + results_df.loc[1, 'BERT'] + results_df.loc[2, 'BERT']).mean()
# S2=np.array(results_df.loc[3, 'BERT'] + results_df.loc[4, 'BERT'] + results_df.loc[5, 'BERT']).mean()
# S3=np.array(results_df.loc[6, 'BERT'] + results_df.loc[7, 'BERT'] + results_df.loc[8, 'BERT']).mean()
# print(S1,S2,S3,'BERT')

results_df

563
563
563
=
1689


,subject,WER,BLEU,METEOR,BERT
0,1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,2,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [4]:
results_df

,subject,WER,BLEU,METEOR,BERT
0,1,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,2,"[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,3,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [5]:
S1=np.array(results_df.loc[0, 'BERT']).mean()
S2=np.array(results_df.loc[1, 'BERT']).mean()
S3=np.array(results_df.loc[2, 'BERT']).mean()
to_file = pd.DataFrame({'subject':[1,2,3], 'significantly_decoded': [S1,S2,S3]})
to_file.to_csv('perceived_speech_percentages.csv', index=False)

to_file

,subject,significantly_decoded
0,1,0.619893
1,2,0.642984
2,3,0.808171


In [6]:
results = np.load('results/S1/perceived_speech/wheretheressmoke.npz', allow_pickle=True)
result_names = results.files
result_files={}
for name in result_names:
    result_files[name] = results[name]
result_files

{'words': array(['she', 'said', 'she', ..., 'that', 'she', 'needs'],
       shape=(1589,), dtype='<U13'),
 'times': array([ 10.2,  10.6,  11. , ..., 591. , 591.4, 591.8], shape=(1589,))}